## Plot top 5 first level topics number of articles yearly

## Plot top 50 third topics number of articles yeartly grouped by second level topic

In [21]:
from pathlib import Path
from collections import Counter
import plotly.graph_objects as go

import pandas as pd

In [22]:
DATA_DIR = Path("data/")
COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]

In [23]:
files = sorted(DATA_DIR.glob("articles_file_[0-9][0-9].tsv"))
if not files:
    raise FileNotFoundError(f"No articles_file_YY.tsv files in {DATA_DIR}")

# Aggregate paths instead of keeping all articles in memory.
path_counts = Counter()
yearly_counts = Counter()
yearly_topic3_counts = Counter()
topic_labels = {}

for file in files:
    print(f"Reading {file.name}")

    for chunk in pd.read_csv(
        file,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        chunksize=100_000,
    ):
        chunk.columns = chunk.columns.str.strip()

        missing = set(COLUMNS) - set(chunk.columns)
        if missing:
            raise ValueError(f"{file.name}: missing columns {sorted(missing)}")

        for column in COLUMNS:
            chunk[column] = chunk[column].str.strip()

        # Example: wissenschaft,Wissenschaft; forschungsmagazin,Forschung: Das Magazin
        if "Topic_Labels" in chunk.columns:
            for value in chunk["Topic_Labels"].unique():
                for item in value.split(";"):
                    slug, separator, label = item.partition(",")
                    if separator and slug.strip() and label.strip():
                        topic_labels.setdefault(slug.strip(), label.strip())

        # A path needs a first-level topic.
        valid = chunk.loc[chunk["Topic_1"].ne(""), COLUMNS]
        counts = valid.groupby(COLUMNS, sort=False).size()
        path_counts.update(counts.to_dict())

        # Keep only the small year/topic aggregate needed for the time-series plot.
        yearly = chunk.loc[
            chunk["Topic_1"].ne("") & chunk["Timestamp"].ne(""),
            ["Topic_1", "Timestamp"],
        ].copy()
        yearly["Year"] = pd.to_datetime(
            pd.to_numeric(yearly["Timestamp"], errors="coerce"),
            unit="s",
            errors="coerce",
            utc=True,
        ).dt.year
        yearly = yearly.dropna(subset=["Year"])
        yearly_counts.update(
            (int(year), topic)
            for year, topic in yearly[["Year", "Topic_1"]].itertuples(index=False, name=None)
        )

        topic3_yearly = chunk.loc[
            chunk["Topic_1"].ne("")
            & chunk["Topic_3"].ne("")
            & chunk["Timestamp"].ne(""),
            ["Topic_1", "Topic_3", "Timestamp"],
        ].copy()
        topic3_yearly["Year"] = pd.to_datetime(
            pd.to_numeric(topic3_yearly["Timestamp"], errors="coerce"),
            unit="s",
            errors="coerce",
            utc=True,
        ).dt.year
        topic3_yearly = topic3_yearly.dropna(subset=["Year"])
        yearly_topic3_counts.update(
            (int(year), topic_1, topic_3)
            for year, topic_1, topic_3 in topic3_yearly[
                ["Year", "Topic_1", "Topic_3"]
            ].itertuples(index=False, name=None)
        )
if not path_counts:
    raise ValueError("No articles with a nonempty Topic_1 were found.")
if not yearly_counts:
    raise ValueError("No articles with a valid year and nonempty Topic_1 were found.")

Reading articles_file_13.tsv
Reading articles_file_14.tsv
Reading articles_file_15.tsv
Reading articles_file_16.tsv
Reading articles_file_17.tsv
Reading articles_file_18.tsv
Reading articles_file_19.tsv
Reading articles_file_20.tsv
Reading articles_file_21.tsv
Reading articles_file_22.tsv


### 4. Evolution of the top five first-level topics

The five topics are selected by their total article count across all years. The chart then shows how many articles belonged to each selected `Topic_1` in each year.

In [17]:
# Select the five first-level topics by total article count.
first_level_totals = Counter()
for (year, topic), count in yearly_counts.items():
    first_level_totals[topic] += count

top5_topics = [topic for topic, _ in first_level_totals.most_common(5)]
years = sorted({year for year, _ in yearly_counts})
yearly_top5 = pd.DataFrame(
    [
        {
            "Year": year,
            "Topic": topic,
            "Articles": yearly_counts.get((year, topic), 0),
        }
        for year in years
        for topic in top5_topics
    ]
)
# yearly_top5["Topic"] = yearly_top5["Topic"].map(
#     lambda topic: DISPLAY_NAMES.get(topic, topic_labels.get(topic, topic))
# )

print("Top 5 first-level topics:")
print(
    yearly_top5.groupby("Topic", sort=False)["Articles"]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)
yearly_top5.head(30)

Top 5 first-level topics:
Topic
web              75463
international    68180
wirtschaft       61344
sport            61082
panorama         60912


,Year,Topic,Articles
0,2013,web,9270
1,2013,international,7541
2,2013,wirtschaft,10597
3,2013,sport,6362
4,2013,panorama,7324
5,2014,web,8928
6,2014,international,7458
7,2014,wirtschaft,7641
8,2014,sport,6016
9,2014,panorama,7219


In [20]:
fig = go.Figure()
for topic in yearly_top5["Topic"].drop_duplicates():
    series = yearly_top5.loc[yearly_top5["Topic"].eq(topic)]
    fig.add_trace(
        go.Scatter(
            x=series["Year"],
            y=series["Articles"],
            mode="lines+markers",
            name=topic,
            hovertemplate="%{x}: %{y:,} articles<extra>%{fullData.name}</extra>",
        )
    )

fig.update_layout(
    title="Evolution of the top five first-level topics",
    xaxis_title="Year",
    yaxis_title="Number of articles",
    hovermode="x unified",
    template="plotly_white",
    height=550,
    legend_title="Topic_1",
)
fig.show()


### 5. Evolution of the top 50 third-level topics

Topics are ranked by total article count across all years. Each line is colored by its first-level parent topic.

In [31]:
topic3_totals = Counter()
topic3_parent = {}
for (year, parent, topic3), count in yearly_topic3_counts.items():
    topic3_totals[topic3] += count
    topic3_parent.setdefault(topic3, parent)

top50_topic3 = [topic for topic, _ in topic3_totals.most_common(100)]
topic3_years = sorted({year for year, _, topic in yearly_topic3_counts if topic in top50_topic3})
yearly_top50_topic3 = pd.DataFrame(
    [
        {
            "Year": year,
            "Topic_3": topic3,
            "Topic_1": topic3_parent[topic3],
            "Articles": yearly_topic3_counts.get((year, topic3_parent[topic3], topic3), 0),
        }
        for year in topic3_years
        for topic3 in top50_topic3
    ]
)
print("Top 50 third-level topics:")
print(
    yearly_top50_topic3.groupby(["Topic_1", "Topic_3"], sort=False)["Articles"]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)
yearly_top50_topic3.head()

Top 50 third-level topics:
Topic_1        Topic_3                      
lifestyle      sudoku                           6014
international  usa                              5715
diskurs        kommentare-der-anderen           5336
sport          bundesliga                       4514
lifestyle      reiseaktuell                     4495
wirtschaft     banken                           4013
international  deutschland                      3715
etat           switchlist                       3312
sport          fussballinternational            3182
etat           tvtagebuch                       3080
web            apple                            3056
diskurs        rauscher                         2985
international  syrien                           2914
web            android                          2900
international  tuerkei                          2576
               ukraine                          2574
sport          deutscheliga                     2433
panorama       flucht-und-p

,Year,Topic_3,Topic_1,Articles
0,2013,sudoku,lifestyle,605
1,2013,usa,international,390
2,2013,kommentare-der-anderen,diskurs,505
3,2013,bundesliga,sport,487
4,2013,reiseaktuell,lifestyle,528


In [32]:
parent_topics = yearly_top50_topic3["Topic_1"].drop_duplicates().tolist()
parent_palette = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00",
    "#56B4E9", "#F0E442", "#882255", "#44AA99", "#AA4499",
]
parent_colors = {
    parent: parent_palette[index % len(parent_palette)]
    for index, parent in enumerate(parent_topics)
}

fig = go.Figure()
for topic3 in top50_topic3:
    series = yearly_top50_topic3.loc[yearly_top50_topic3["Topic_3"].eq(topic3)]
    parent = topic3_parent[topic3]
    fig.add_trace(
        go.Scatter(
            x=series["Year"],
            y=series["Articles"],
            mode="lines",
            name=topic3,
            legendgroup=parent,
            line={"color": parent_colors[parent], "width": 1.5},
            hovertemplate=(
                "%{x}: %{y:,} articles<extra>"
                + topic3
                + " ("
                + parent
                + ")</extra>"
            ),
            showlegend=True,
        )
    )

for parent, color in parent_colors.items():
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="lines",
            line={"color": color, "width": 4},
            name=f"First level: {parent}",
            legendgroup=parent,
            showlegend=True,
        )
    )

fig.update_layout(
    title="Evolution of the top 50 third-level topics",
    xaxis_title="Year",
    yaxis_title="Number of articles",
    hovermode="x unified",
    template="plotly_white",
    height=900,
    legend_title="Topic_3 / parent Topic_1",
)
fig.show()
